# Stage 3: Measure our subtraction model from Stage 2

## Rung 0 — Setup & load
Load the difference image (the ONLY thing we detect on) + the aligned science &
template frames (held only for triplet panels later). Crop all to the same central
1000×1000 cutout, carrying the sliced WCS.

Code Block 1: Import all our data

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import config 

import numpy as np 
from astropy.io import fits
from astropy.wcs import WCS
from astropy.nddata import Cutout2D
import matplotlib.pyplot as plt 

#File paths (relative to project root)
DIFF_PATH = "../ztfdata/difference/ztf_20180322273264_000468_zr_c03_o_q2_official_diff.fits"
SCI_PATH  = "../ztfdata/aligned/ztf_20180322273264_000468_zr_c03_o_q2_sciimg.fits"  # science epoch
REF_PATH  = "../ztfdata/aligned/ztf_20180409218044_000468_zr_c03_o_q2_sciimg.fits"  # template (sharpest, idx 2)

#load each as (data, wcs); data in HDU 0 for all 3 paths defined above
def load_fits(path):
    with fits.open(path) as hdul:
        data = hdul[0].data.astype(float)
        wcs = WCS(hdul[0].header)
    return data, wcs

diff_full, diff_wcs = load_fits(DIFF_PATH)
#after result
sci_full, sci_wcs = load_fits(SCI_PATH)
#before result
ref_full, ref_wcs = load_fits(REF_PATH)

print("full-frame shape:", diff_full.shape)

Cell Block 2: Importing central cutout (1000x1000 region made in stage 2)

In [ ]:
CUTOUT_SIZE = 1000
center = (diff_full.shape[1] // 2, diff_full.shape[0] // 2)

diff_cut = Cutout2D(diff_full, center, CUTOUT_SIZE, wcs=diff_wcs)
sci_cut = Cutout2D(sci_full, center, CUTOUT_SIZE, wcs=diff_wcs)
ref_cut = Cutout2D(ref_full, center, CUTOUT_SIZE, wcs=ref_wcs)

diff = diff_cut.data
cut_wcs = diff_cut.wcs

print("cutout shape:", diff.shape, "| NaNs in diff cutout:", np.isnan(diff).sum())

Cell Block 3: Eyeballing inputs

In [ ]:
# --- eyeball inputs: difference (what we detect on) vs science (has static stars) ---
from astropy.visualization import ZScaleInterval

z = ZScaleInterval()   # standard astronomy stretch: picks black/white from the data itself

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# DIFFERENCE: keep the tight symmetric range — a diff is signed noise (~±30) around 0
axes[0].imshow(diff, origin="lower", cmap="gray", vmin=-30, vmax=30)
axes[0].set_title("DIFFERENCE — the figure lives here (residuals)")

# SCIENCE: let ZScale choose the limits from the actual pixel distribution
svmin, svmax = z.get_limits(sci_cut.data)
axes[1].imshow(sci_cut.data, origin="lower", cmap="gray", vmin=svmin, vmax=svmax)
axes[1].set_title("SCIENCE — full of static stars (the ground we cancelled)")

for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()



## Rung 2: Background and noise map

Cell Block 4: background + noise map

In [ ]:
from astropy.stats import sigma_clip
from astropy.stats import SigmaClip
from photutils.background import Background2D, MedianBackground

sigma_clip = SigmaClip(sigma=3.0)
bkg_estimator = MedianBackground()

bkg = Background2D(
    diff,
    box_size=(64,64),
    filter_size=(3,3),
    sigma_clip=sigma_clip,
    bkg_estimator=bkg_estimator,
)

rms = bkg.background_rms

print(f"median background : {np.median(bkg.background):.3f}   (expect ~0 for a diff image)")
print(f"median noise (RMS): {np.median(rms):.3f}   (compare to Stage-2 global std ~10.9)")

Cell Block 5: See the Ruler

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(diff, origin="lower", cmap="gray", vmin=-30, vmax=30)
axes[0].set_title("DIFFERENCE")
im = axes[1].imshow(rms, origin="lower", cmap="viridis")
axes[1].set_title("NOISE (RMS) MAP — the ruler")
fig.colorbar(im, ax=axes[1], fraction=0.046)
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()


## Rung 2: Detect both signs. Stuff that appears/disappears

Cell Block 6: the actual detection being detected

In [ ]:
from photutils.segmentation import detect_sources


N_SIGMA = 3.0
NPIXELS = 5

threshold = N_SIGMA * rms

segm_pos = detect_sources(diff, threshold, npixels=NPIXELS)
segm_neg = detect_sources(-diff, threshold, npixels=NPIXELS)

n_pos = segm_pos.nlabels if segm_pos is not None else 0
n_neg = segm_neg.nlabels if segm_neg is not None else 0

print(f"N = {N_SIGMA}sigma  ->  positive blobs: {n_pos}   negative blobs: {n_neg}   total: {n_pos + n_neg}")

Cell Block 7: see what got detected

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].imshow(diff, origin="lower", cmap="gray", vmin=-30, vmax=30)
axes[0].set_title("DIFFERENCE")

# segmentation maps: each blob a colored patch (background transparent-ish)
axes[1].imshow(diff, origin="lower", cmap="gray", vmin=-30, vmax=30)
if segm_pos is not None:
    axes[1].contour(segm_pos.data > 0, levels=[0.5], colors="lime",    linewidths=0.6)
if segm_neg is not None:
    axes[1].contour(segm_neg.data > 0, levels=[0.5], colors="magenta", linewidths=0.6)
axes[1].set_title(f"DETECTIONS  (green=+ {n_pos},  magenta=- {n_neg})  @ {N_SIGMA}sigma")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()


## Rung 3: Deblend to split merged blobs to visually see difference

Cell Block 8: Deblending

In [ ]:
from photutils.segmentation import deblend_sources

def deblend(data, segm):
    if segm is None:
        return None
    return deblend_sources(data, segm, n_pixels=NPIXELS, n_levels=32, contrast=0.001)

segm_pos_db = deblend(diff,  segm_pos)   # deblend the positive map
segm_neg_db = deblend(-diff, segm_neg)   # deblend the negative map (matches Rung 2's -diff)

def nlab(s):
    return s.n_labels if s is not None else 0

print("positives:  before deblend", nlab(segm_pos),  "->  after", nlab(segm_pos_db))
print("negatives:  before deblend", nlab(segm_neg),  "->  after", nlab(segm_neg_db))
print("TOTAL sources after deblend:", nlab(segm_pos_db) + nlab(segm_neg_db))


## Rung 4: Measure
what we're measuring: 
- Position
- Brightness
- Shape

Cell block #9: Measure every source

In [ ]:
# --- Rung 4: measure every source -> one combined, sign-tagged catalog ---
from photutils.segmentation import SourceCatalog
from astropy.table import vstack

COLUMNS = [
    "label",
    "x_centroid", "y_centroid",          # sub-pixel center (pixels) -- renamed in photutils 3.0
    "sky_centroid",                      # RA/Dec object (from the WCS)
    "segment_flux",
    "max_value",
    "elongation", "ellipticity", "orientation",   # SHAPE = Stage-4 features
    "area",
]

def measure(data, segm_db, sign):
    if segm_db is None:
        return None
    cat = SourceCatalog(data, segm_db, wcs=cut_wcs, error=rms)
    tbl = cat.to_table(columns=COLUMNS)
    tbl["ra"]  = tbl["sky_centroid"].ra.deg
    tbl["dec"] = tbl["sky_centroid"].dec.deg
    tbl.remove_column("sky_centroid")
    tbl["snr"] = cat.segment_flux / cat.segment_flux_err   # renamed in photutils 3.0
    tbl["sign"] = sign
    return tbl

tbl_pos = measure(diff,  segm_pos_db, +1)
tbl_neg = measure(-diff, segm_neg_db, -1)

catalog = vstack([t for t in (tbl_pos, tbl_neg) if t is not None])
print(f"catalog rows: {len(catalog)}   (pos {0 if tbl_pos is None else len(tbl_pos)}, "
      f"neg {0 if tbl_neg is None else len(tbl_neg)})")
catalog[:8]




## Rung 5: Cutout Differences

Cell Block 10: Cutouts and saving them to project

In [ ]:
import os 
from astropy.nddata import Cutout2D

STAMP = 63
CUTOUTS_DIR = "../ztfdata/ztf_own_pipeline_data_testing_small_scale/cutouts" 
os.makedirs(CUTOUTS_DIR, exist_ok=True)

sci_img = sci_cut.data
ref_img = ref_cut.data
diff_img = diff

def make_triplet(x,y):
    pos = (x,y)
    kw = dict(size=STAMP, mode="partial", fill_value = 0.0)
    s = Cutout2D(sci_img, pos, **kw).data
    r = Cutout2D(ref_img, pos, **kw).data
    d = Cutout2D(diff_img, pos, **kw).data
    return np.stack([s,r,d], axis=0)

stamp_paths =[]
half = STAMP / 2
H, W = diff.shape
for row in catalog:
    x, y = row["x_centroid"], row["y_centroid"]
    triplet = make_triplet(x,y)

    on_edge = (x < half) or (y < half) or (x > W - half) or (y > H - half)
    fname = f"src_{row['sign']:+d}_{int(row['label']):03d}.npy".replace("+", "p").replace("-", "m")
    fpath = os.path.join(CUTOUTS_DIR, fname)
    np.save(fpath, triplet)
    stamp_paths.append((fpath, on_edge))

catalog["stamp_path"] = [p for p, _ in stamp_paths]
catalog["on_edge"]    = [e for _, e in stamp_paths]

print(f"saved {len(stamp_paths)} triplets to {CUTOUTS_DIR}/")
print(f"  edge-affected (zero-padded): {sum(e for _, e in stamp_paths)}")
print(f"  each stamp shape: (3, {STAMP}, {STAMP})  channels = [sci, ref, diff]")


Cell Block 11: visualizatin of a single triplet

In [ ]:
from astropy.visualization import ZScaleInterval
z = ZScaleInterval()

mask = (catalog["label"] == 3) & (catalog["sign"] == 1) 
row = catalog[mask][0]
trip = np.load(row["stamp_path"])

titles = ["SCIENCE (after)", "REFERENCE (before)", "DIFFERENCE (change)"]
fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, panel, t in zip(axes, trip, titles):
    vmin, vmax = z.get_limits(panel)
    ax.imshow(panel, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)
    ax.set_title(t); ax.axis("off")
fig.suptitle(f"Triplet for source label 3  (elongation {row['elongation']:.1f}, SNR {row['snr']:.0f})")
plt.tight_layout(); plt.show()


## Rung 6: Validate and Compare Our Model with That of ZTF

Cell BLock 12: validate our cutouts against a real ZTF alert

In [ ]:
from ztfquery import alert
from astropy.visualization import ZScaleInterval
z = ZScaleInterval()

ALERT_PATH = "../ztfdata/alert_sample/sample_alert.avro" 
reader = alert.AlertReader.load(ALERT_PATH)
cand = reader.alert["candidate"]

# --- their catalog row: the fields that map to ours --
print("=== ZTF alert candidate (professional Stage-3 row) ===")
print(f"  ra, dec     : {cand['ra']:.5f}, {cand['dec']:.5f}")
print(f"  elong       : {cand['elong']:.3f}      (our 'elongation')")
print(f"  fwhm        : {cand['fwhm']:.2f}")
print(f"  isdiffpos   : {cand['isdiffpos']!r}    (our 'sign': f=-1, t=+1)")
print(f"  magpsf      : {cand['magpsf']:.2f} +/- {cand['sigmapsf']:.3f}")
print(f"  rb (real/bogus): {cand['rb']:.3f}   <-- STAGE 4's job; ours stops before this")

# --- their triplet (all 63x63, same as ours) ---
ztf_trip = [reader.get_stamp(w).data for w in ["Science","Template","Difference"]]

# --- one of OUR triplets for side-by-side (the streak candidate, label 3) ---
row = catalog[(catalog["label"]==3) & (catalog["sign"]==1)][0]
our_trip = np.load(row["stamp_path"])   # (3,63,63): sci, ref, diff

# Visualization
fig, axes = plt.subplots(2, 3, figsize=(11, 7.5))
titles = ["Science", "Reference/Template", "Difference"]
for j,(panel,t) in enumerate(zip(ztf_trip, titles)):
    vmin,vmax = z.get_limits(panel)
    axes[0,j].imshow(panel, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)
    axes[0,j].set_title(f"ZTF alert — {t}")
for j,(panel,t) in enumerate(zip(our_trip, titles)):
    vmin,vmax = z.get_limits(panel)
    axes[1,j].imshow(panel, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)
    axes[1,j].set_title(f"OURS (src 3) — {t}")
for ax in axes.ravel(): ax.axis("off")
fig.suptitle("Professional ZTF triplet (top)  vs  our Stage-3 triplet (bottom) — both 63x63")
plt.tight_layout(); plt.show()


Cell Block 13: handoff to stage 4 judging phase

In [ ]:
import os
from astropy.table import Table

CATALOGS_DIR = "../ztfdata/ztf_own_pipeline_data_testing_small_scale/catalog"
os.makedirs(CATALOGS_DIR, exist_ok=True)
CATALOG_PATH = os.path.join(CATALOGS_DIR, "ztf_20180322273264_000468_zr_c03_o_q2_stage3_catalog.ecsv")

catalog.write(CATALOG_PATH, format="ascii.ecsv", overwrite=True)
print(f"wrote {len(catalog)} sources -> {CATALOG_PATH}")

reloaded = Table.read(CATALOG_PATH)
print(f"re-loaded OK: {len(reloaded)} rows, {len(reloaded.colnames)} columns")
print("columns:", reloaded.colnames)
